<a href="https://colab.research.google.com/github/Shrideshi1/multi-label-email-risk-detection/blob/main/notebooks/03_2_model_training_CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##Libraries
import os
import joblib
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from google.colab import drive, userdata
from sklearn.feature_extraction.text import TfidfVectorizer
##Cross Validation
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, hamming_loss
import numpy as np

In [ ]:
drive.mount("/content/drive")

##Project Directory
PROJECT_DIR = "/content/drive/.shortcut-targets-by-id/1SLhiH7VulyiPZ7L846kJBEbN1pa4c4zU/Multi_Label_Email_Risk_Detection/"
DATA_PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"
MODELS_DIR = f"{PROJECT_DIR}/models"
REPORTS_DIR = f"{PROJECT_DIR}/reports"
FEATURE_DIR = f"{PROJECT_DIR}/data/processed/features"
os.chdir(PROJECT_DIR)

print("Current folder:", os.getcwd())
print("Processed files:", os.listdir(DATA_PROCESSED_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current folder: /content/drive/.shortcut-targets-by-id/1SLhiH7VulyiPZ7L846kJBEbN1pa4c4zU/Multi_Label_Email_Risk_Detection
Processed files: ['combined_dataset.csv', 'features', 'synthetic_confidential.csv']


In [ ]:
combined_df = pd.read_csv(f"{DATA_PROCESSED_DIR}/combined_dataset.csv")

risk_cols = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]

print("Dataset shape:", combined_df.shape)
display(combined_df.head())

Dataset shape: (69910, 12)


,text,category,source,financial_risk,credential_risk,customer_info_risk,proprietary_risk,legal_risk,attachment_risk,phishing_spam_risk,project_category,risk_score
0,job posting - apple-iss research center conten...,ham,messages,0,0,0,0,0,0,0,Ham,0
1,"lang classification grimes , joseph e . and b...",ham,messages,0,0,0,0,0,0,0,Ham,0
2,query : letter frequencies for text identifica...,ham,messages,0,0,0,0,0,0,0,Ham,0
3,risk a colleague and i are researching the dif...,ham,messages,0,0,0,0,0,0,0,Ham,0
4,request book information earlier this morning ...,ham,messages,0,0,0,0,0,0,0,Ham,0


In [ ]:
TEXT_COLUMN = "text"

X_text = combined_df[TEXT_COLUMN].fillna("").astype(str)
y = combined_df[risk_cols].values

##Vectorize text using TF-IDF to create feature matrix X
vectorizer = TfidfVectorizer()
X_features = vectorizer.fit_transform(X_text)

print(f"Feature matrix shape (X): {X_features.shape}")
print(f"Target label matrix shape (y): {y.shape}")

Feature matrix shape (X): (69910, 221171)
Target label matrix shape (y): (69910, 7)


In [ ]:
##K-Fold CV
kf = KFold(n_splits=10, shuffle=True, random_state=26)

##Define Models
models_to_evaluate = {
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "Random Forest": MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=26)),
    "XGBoost": MultiOutputClassifier(XGBClassifier(eval_metric='logloss', random_state=26)),
    "Linear SVM": MultiOutputClassifier(LinearSVC(random_state=26))
}
##Empty for results
cv_summary = {}

In [ ]:
for name, model in models_to_evaluate.items():
    fold_f1s = []
    fold_hamming = []
    print(f"\nEvaluating: {name} via 10-Fold Cross-Validation...")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_features)):
        X_train, X_val = X_features[train_idx], X_features[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        ##Fit model on fold
        model.fit(X_train, y_train)

        ##Predict on fold
        preds = model.predict(X_val)

        ##Metrics
        f1 = f1_score(y_val, preds, average='macro', zero_division=0)
        h_loss = hamming_loss(y_val, preds)

        fold_f1s.append(f1)
        fold_hamming.append(h_loss)

    cv_summary[name] = {
        "Mean Macro F1": np.mean(fold_f1s),
        "Std Macro F1": np.std(fold_f1s),
        "Mean Hamming Loss": np.mean(fold_hamming)
    }


Evaluating: Logistic Regression via 10-Fold Cross-Validation...

Evaluating: Random Forest via 10-Fold Cross-Validation...

Evaluating: XGBoost via 10-Fold Cross-Validation...

Evaluating: Linear SVM via 10-Fold Cross-Validation...


In [ ]:
##Results
results_table = pd.DataFrame(cv_summary).T
display(results_table)

,Mean Macro F1,Std Macro F1,Mean Hamming Loss
Logistic Regression,0.981151,0.002836,0.003210
Random Forest,0.990674,0.001581,0.002066
XGBoost,0.990633,0.001830,0.002385
Linear SVM,0.994231,0.001031,0.001101


In [ ]:

def evaluate_new_email(email_text_string):
    """
    Takes raw email text, applies the TF-IDF vectorizer,
    and predicts risk flags across all risk columns.
    """
    ##Test into Vectorized form
    new_email_vectorized = vectorizer.transform([email_text_string])

    comparison_data = []

    for name, model in models_to_evaluate.items():
        ##Risk flags for all 7 categories
        prediction = model.predict(new_email_vectorized)[0]

        ##Risk column with its corresponding binary prediction (0 or 1)
        for risk_col, pred_val in zip(risk_cols, prediction):
            comparison_data.append({
                "Model": name,
                "Risk Category": risk_col,
                "Predicted Risk (0/1)": pred_val
            })

    results_df = pd.DataFrame(comparison_data)

    return results_df

In [ ]:
sample = """
Subject: Confidential Q3 Financial Report and System Access Information

Hi Team,

Please review the attached confidential Q3 financial report before the executive meeting. The document contains internal revenue forecasts, budget projections, customer account information, and employee payroll details.

The updated contract agreement with our partner includes confidential legal terms, pricing information, and compliance requirements. This information is classified as internal use only and should not be forwarded or distributed outside the company.

Please use the secure portal credentials below to access the report:

Username: finance_admin
Password: TempPassword123!
API Key: sk_live_83jd92kd92

The attached files include:
- Q3_Financial_Report.xlsx
- Customer_Database_Update.csv
- Partnership_Agreement.pdf

Please verify the payment details for the upcoming $250,000 vendor invoice and confirm the transaction status.

Do not share this email, attachments, or credentials with unauthorized users.

Regards,
John
Finance Department
"""

print("Evaluating incoming test email...")
risk_report = evaluate_new_email(sample)
display(risk_report)

Evaluating incoming test email...


,Model,Risk Category,Predicted Risk (0/1)
0,Logistic Regression,financial_risk,0
1,Logistic Regression,credential_risk,0
2,Logistic Regression,customer_info_risk,0
3,Logistic Regression,proprietary_risk,0
4,Logistic Regression,legal_risk,0
5,Logistic Regression,attachment_risk,0
6,Logistic Regression,phishing_spam_risk,0
7,Random Forest,financial_risk,0
8,Random Forest,credential_risk,0
9,Random Forest,customer_info_risk,0
